# LRA quality cutoffs — picking the long-read-assembly tier

Three tiers of assemblies, overlaid on the same QC distributions, so we can choose where to draw the cutoff that defines the **accepted LRA set** (Complete-Genome ∪ accepted near-complete GCAs):

- **Pale blue — `is_refseq`**: every RefSeq genome already in the curated metadata. The reference of what 'good enough to publish as a reference' looks like.
- **Purple, α=0.5 — `LR-GCA · Complete Genome`**: the 1,202 long-read GCAs whose NCBI `assembly_level == "Complete Genome"`. We accept these by definition.
- **Red, α=0.5 — `LR-GCA · non-complete`**: the 1,369 long-read GCAs at Chromosome / Scaffold / Contig level — candidates for inclusion if they're near-complete.

QC is uniform across all three tiers: completeness, contamination, n_contigs, contig_n50 all come from **NCBI Datasets v2 `dataset_report`** (the curated metadata's bakrep QC has no CheckM populated for the is_refseq rows, so we fetch from NCBI for everything — one consistent source). Cached to `src/bac_data/lr_data/notebooks/_data/`.

Decision artefact: the final cell records the chosen acceptance rule (thresholds + boolean). Those values become module constants in `src/bac_data/lr_data/build_lra_set.py`.

## 0 · Parameters — edit these to re-tune the LRA cutoff

`USE_REFSEQ_PERCENTILE` controls how the four RefSeq-derived thresholds (completeness, contamination, n_contigs, contig_n50) are drawn from the RefSeq distribution (Table A in §9b).

- `0` — absolute min/max; **zero RefSeq excluded** by construction. Most inclusive of LR-GCAs.
- `1` or `2` — trim the worst 1 % / 2 % of RefSeq from each tail; tighter rule, excludes those RefSeq outliers too.

`MIN_LR_COVERAGE` is a hand-set LR-only floor (no RefSeq comparator — none have an `est_coverage_5_5Mb`).

In [ ]:
# ─── PARAMETERS ────────────────────────────────────────────────────────────
USE_REFSEQ_PERCENTILE = 0     # 0 | 1 | 2 — % trimmed from each tail of RefSeq distribution
MIN_LR_COVERAGE       = 20.0  # × (LR-only floor; no RefSeq comparator)
# ───────────────────────────────────────────────────────────────────────────

In [ ]:
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make the in-repo bac_data package importable so we can reuse the NCBI fetcher.
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.name and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

# Auto-detect data root: HPC path first, then local OneDrive mirror.
def _first_existing(candidates):
    for c in candidates:
        p = Path(c)
        if p.exists():
            return p
    raise FileNotFoundError(f"None of: {candidates}")

RELATED_LR_DIR = _first_existing([
    "/home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/raw/related_lr",
    "/Users/davidabelson/Library/CloudStorage/OneDrive-UniversityofCambridge/local_data/klebsiella/raw/related_lr",
])
METADATA_TSV = _first_existing([
    "/home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/final/metadata_final_curated_all_samples_and_columns.tsv",
    "/Users/davidabelson/Library/CloudStorage/OneDrive-UniversityofCambridge/Aaron Weimann's files - project_k/data/final/metadata/metadata_final_curated_all_samples_and_columns.tsv",
])
RELATED_LR_RUNS_CSV = _first_existing([
    "/home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/final/related_lr_run_accessions.csv",
    "/Users/davidabelson/Library/CloudStorage/OneDrive-UniversityofCambridge/Aaron Weimann's files - project_k/data/final/metadata/related_lr_run_accessions.csv",
])

# Notebook now lives at src/bac_data/lr_data/notebooks/; cache is sibling _data/.
CACHE_DIR = REPO_ROOT / "src" / "bac_data" / "lr_data" / "notebooks" / "_data"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
NCBI_LR_CACHE     = CACHE_DIR / "lra_quality_metrics_lr.tsv"
NCBI_REFSEQ_CACHE = CACHE_DIR / "lra_quality_metrics_refseq.tsv"

# Tier palette: pale blue baseline, purple complete LR, red non-complete LR.
TIER_COLOURS = {
    "is_refseq":            ("#9ec5e8", 0.5),  # pale blue
    "LR · Complete Genome": ("#7b3fbf", 0.5),  # purple
    "LR · non-complete":    ("#d62728", 0.5),  # red
}
TIER_ORDER = list(TIER_COLOURS)

# Re-used by the fetch cell; resolves to the right module pre/post Phase C.
def _import_fetcher():
    try:
        from bac_data.lr_data.gca_to_gcf_lookup import lookup_all_gcas  # post Phase C
    except ImportError:
        from bac_data.gca_to_gcf_lookup import lookup_all_gcas          # pre Phase C
    return lookup_all_gcas

print(f"RELATED_LR_DIR     = {RELATED_LR_DIR}")
print(f"METADATA_TSV       = {METADATA_TSV}")
print(f"RELATED_LR_RUNS    = {RELATED_LR_RUNS_CSV}")
print(f"NCBI_LR_CACHE      = {NCBI_LR_CACHE}")
print(f"NCBI_REFSEQ_CACHE  = {NCBI_REFSEQ_CACHE}")

## 1 · Load the LR-GCA discovery table

In [ ]:
lr_gca = pd.read_csv(RELATED_LR_DIR / "related_lr_all_gca.tsv", sep="\t", low_memory=False)
lr_gca["gca_bare"] = lr_gca["gca"].astype(str).str.split(".").str[0]
lr_gca["has_gcf"] = lr_gca["gcf"].fillna("").astype(str).str.startswith("GCF_")
lr_gca["is_complete_level"] = lr_gca["level"] == "Complete Genome"
print(f"LR-GCAs: {len(lr_gca)}")
print(lr_gca["level"].value_counts())
lr_gca.head(3)

## 2 · Fetch CheckM + scaffold/contig stats for the LR GCAs (cached)

Reuses `bac_data.gca_to_gcf_lookup.lookup_all_gcas` (the batched, rate-limited NCBI Datasets v2 fetcher). Cached to `src/bac_data/lr_data/notebooks/_data/lra_quality_metrics_lr.tsv` — delete that file to refetch.

Set `NCBI_API_KEY` in the environment (10 req/s vs 3) before launching jupyter.

In [ ]:
if NCBI_LR_CACHE.exists():
    ncbi_metrics = pd.read_csv(NCBI_LR_CACHE, sep="\t", low_memory=False)
    print(f"Loaded cache: {len(ncbi_metrics)} rows from {NCBI_LR_CACHE.name}")
else:
    lookup_all_gcas = _import_fetcher()
    print(f"No cache; fetching {lr_gca['gca_bare'].nunique()} GCAs from NCBI Datasets …")
    ncbi_metrics = lookup_all_gcas(lr_gca["gca_bare"].dropna().unique().tolist(), batch_size=100)
    ncbi_metrics.to_csv(NCBI_LR_CACHE, sep="\t", index=False)
    print(f"Wrote cache: {NCBI_LR_CACHE} ({len(ncbi_metrics)} rows)")

# Merge NCBI metrics onto the LR-GCA frame (bare-accession join).
lr_gca_q = lr_gca.merge(
    ncbi_metrics.rename(columns={"lookup_accession": "gca_bare"}),
    on="gca_bare", how="left",
)
n_total   = len(lr_gca_q)
n_checkm  = lr_gca_q["ncbi_completeness"].notna().sum()
print(f"NCBI metrics fetched for {n_total} / {n_total} LR-GCAs")
print(f"  with CheckM:    {n_checkm} ({100 * n_checkm / n_total:.0f}%)")
print(f"  without CheckM: {n_total - n_checkm} ({100 * (n_total - n_checkm) / n_total:.0f}%)  ← see §4b")

# Preview the populated case (head(3) of the raw merge happened to land on
# Contig-level SKESA rows that all lack CheckM — show a CheckM-present sample).
lr_gca_q.dropna(subset=["ncbi_completeness"]).head(3)[
    ["gca", "level", "ncbi_completeness", "ncbi_contamination", "ncbi_contig_n50", "ncbi_n_contigs"]
]

## 3 · Attach LR-run coverage to each LR-GCA

`related_lr_run_accessions.csv` already has a precomputed `est_coverage_5_5Mb` column — join on `related_lr_accession` ↔ `run_accession`.

In [ ]:
lr_runs = pd.read_csv(RELATED_LR_RUNS_CSV, low_memory=False)
print(f"LR runs: {len(lr_runs)} rows; with coverage est: {lr_runs['est_coverage_5_5Mb'].notna().sum()}")

lr_gca_q = lr_gca_q.merge(
    lr_runs[["run_accession", "base_count", "est_coverage_5_5Mb"]].rename(
        columns={"run_accession": "related_lr_accession", "est_coverage_5_5Mb": "lr_coverage_est"}
    ),
    on="related_lr_accession", how="left",
)
print(f"LR coverage matched for {lr_gca_q['lr_coverage_est'].notna().sum()} / {len(lr_gca_q)} GCAs")

## 4 · `is_refseq` baseline — extract GCFs from `Sample` and fetch NCBI CheckM (cached)

The is_refseq rows hold GCF accessions in the `Sample` column (e.g. `GCF_020526085.1_ASM2052608v1_genomic`). The curated metadata has bakrep `contig_count` / `N50` for ~90 % of them but **no CheckM** (CheckM2 was only run on our short-read assemblies, not on external RefSeq genomes). So we fetch NCBI's CheckM v1 + assembly stats for the is_refseq GCFs via the same `lookup_all_gcas` helper — uniform QC source across all three tiers.

In [ ]:
REFSEQ_COLS = ["Sample", "is_refseq"]
meta = pd.read_csv(METADATA_TSV, sep="\t", usecols=REFSEQ_COLS, low_memory=False)
refseq = meta.loc[meta["is_refseq"].fillna(False).astype(bool)].copy()

# Sample looks like "GCF_020526085.1_ASM2052608v1_genomic" or "GCA_041863035.1".
# A small fraction of is_refseq rows hold a GCA accession (no RefSeq surrogate
# minted yet) — accept both prefixes; NCBI Datasets v2 takes either.
_ACC_RE = re.compile(r"(GC[AF]_\d+\.\d+)")
refseq["assembly_acc"] = refseq["Sample"].astype(str).str.extract(_ACC_RE, expand=False)
refseq["assembly_acc_bare"] = refseq["assembly_acc"].str.split(".").str[0]
parsed = refseq["assembly_acc"].notna().sum()
n_gca = refseq["assembly_acc"].str.startswith("GCA_", na=False).sum()
n_gcf = refseq["assembly_acc"].str.startswith("GCF_", na=False).sum()
print(f"is_refseq rows: {len(refseq)}; assembly accessions parsed: {parsed} "
      f"({n_gcf} GCF + {n_gca} GCA); unparsed: {len(refseq) - parsed}")

# Incremental cache: load what we have, fetch only what's missing.
# `lookup_all_gcas` itself retries transiently-failed accessions until
# convergence — see src/bac_data/gca_to_gcf_lookup.py.
all_accs = set(refseq["assembly_acc_bare"].dropna().unique())
if NCBI_REFSEQ_CACHE.exists():
    refseq_metrics = pd.read_csv(NCBI_REFSEQ_CACHE, sep="\t", low_memory=False)
    cached_accs = set(refseq_metrics["lookup_accession"].dropna().astype(str))
    missing = sorted(all_accs - cached_accs)
    print(f"Loaded cache: {len(refseq_metrics)} rows; missing {len(missing)} accessions")
else:
    refseq_metrics = pd.DataFrame()
    missing = sorted(all_accs)
    print(f"No cache; will fetch all {len(missing)} accessions")

if missing:
    lookup_all_gcas = _import_fetcher()
    new_metrics = lookup_all_gcas(missing, batch_size=100)
    refseq_metrics = pd.concat([refseq_metrics, new_metrics], ignore_index=True)
    refseq_metrics.to_csv(NCBI_REFSEQ_CACHE, sep="\t", index=False)
    print(f"Wrote cache: {NCBI_REFSEQ_CACHE} ({len(refseq_metrics)} rows, +{len(new_metrics)})")

refseq_q = refseq.merge(
    refseq_metrics.rename(columns={"lookup_accession": "assembly_acc_bare"}),
    on="assembly_acc_bare", how="left",
)
print(f"\nNCBI metrics resolved for {refseq_q['ncbi_completeness'].notna().sum()} / {len(refseq_q)} is_refseq rows "
      f"(missing CheckM ≠ fetch failure — see diagnostic in §4a)")

### 4a · Diagnostic — why ~280 RefSeq rows still lack CheckM after the fetch

All 3,911 accessions return data (no fetch failures). But ~280 still come back without CheckM. The pattern is sharp:

- They are **all GCA-prefixed** Sample values that were flagged `is_refseq=True` upstream.
- They have **no `paired_gcf_accession`** — they were never actually promoted into RefSeq.
- ~276 are one large multi-isolate submission batch (`Flye v2.9 + Unicycler v0.5.0`); the rest are flagged `contaminated` etc.

So the underlying issue is a **stale `is_refseq` flag** in the curated metadata — these rows aren't true RefSeq. The data-driven rule rejects them for `checkm_missing`, which is the right outcome (they shouldn't be setting the RefSeq quality envelope). Flagging here so we don't re-investigate; bac_metadata should re-audit `is_refseq` against actual RefSeq presence in a future pass.

In [ ]:
_stale = refseq_q[refseq_q["ncbi_completeness"].isna()].copy()
_stale["acc_prefix"] = _stale["assembly_acc"].str[:4]
print(f"is_refseq rows lacking CheckM: {len(_stale)}")
print(f"\nBy accession prefix:")
print(_stale["acc_prefix"].value_counts())
print(f"\nHow many of these lack paired_gcf_accession (= not in RefSeq):")
print(f"  {_stale['paired_gcf_accession'].isna().sum()} / {len(_stale)}")
print(f"\nTop assembly_method values (stale rows):")
print(_stale["ncbi_assembly_method"].value_counts(dropna=False).head(5))

### 4b · Why is CheckM missing for ~30 % of LR-GCAs?

NCBI does compute CheckM v1 on prokaryotic submissions, but **deliberately skips assemblies it has flagged as anomalous** — contaminated, unverified source organism, oversized genome length, etc. So a NaN `ncbi_completeness` is itself a quality signal: "NCBI thinks this assembly is suspect." The decision rule in §10 treats NaN as a reject for that reason.

What we *do* have for the NaN-CheckM LR-GCAs: the NCBI assembly stats (`n_contigs`, `contig_n50`, `genome_size`, `genome_coverage`) and the submitter's `genome_notes`. The summary below contrasts the with-CheckM vs without-CheckM populations so you can see whether the rejected ones look statistically worse, or just bear a quality flag.

In [ ]:
_stat_cols = ["ncbi_n_contigs", "ncbi_contig_n50", "ncbi_genome_size", "ncbi_genome_coverage"]
lr_gca_q["_has_checkm"] = lr_gca_q["ncbi_completeness"].notna()

# Coerce coverage to numeric (submitter strings like "240" / "100x" → number)
lr_gca_q["_coverage_num"] = pd.to_numeric(
    lr_gca_q["ncbi_genome_coverage"].astype(str).str.replace(r"[^\d\.]", "", regex=True),
    errors="coerce",
)
stat_view = lr_gca_q.copy()
stat_view["ncbi_genome_coverage"] = stat_view["_coverage_num"]

print("=== Median assembly stats: with-CheckM vs without-CheckM (LR-GCA tier) ===")
summary = stat_view.groupby("_has_checkm")[_stat_cols].median().round(0)
summary["n_rows"] = stat_view.groupby("_has_checkm").size()
summary = summary.rename(index={True: "with CheckM", False: "without CheckM"})
display(summary)

print("\n=== Sample of LR-GCAs without CheckM ===")
without = stat_view[~stat_view["_has_checkm"]]
display(without[["gca", "ncbi_n_contigs", "ncbi_contig_n50", "ncbi_genome_size",
                 "ncbi_genome_coverage", "ncbi_genome_notes", "ncbi_sequencing_tech"]].head(10))

# ─── Narrower view: NaN-CheckM AND not-obviously-undersequenced ────────────────
# Most without-CheckM rows have <95× coverage anyway, so they'd be excluded by
# the coverage criterion regardless. Filter to the harder-to-explain subset.
COVERAGE_FLOOR = 95.0
hicov = without[without["ncbi_genome_coverage"] > COVERAGE_FLOOR]
print(f"\n=== NaN-CheckM AND ncbi_genome_coverage > {COVERAGE_FLOOR:.0f}× ===")
print(f"Count: {len(hicov)} / {len(without)} of the without-CheckM rows")
print(f"      ({len(hicov)} / {len(lr_gca_q)} of all LR-GCAs)")

if len(hicov):
    # Inline supplementary fetch for organism_name — not in the cached extract_row.
    import requests
    species_cache = CACHE_DIR / "lra_nan_checkm_hicov_species.tsv"
    accs_now = sorted(hicov["gca_bare"].dropna().unique())
    if species_cache.exists():
        spp = pd.read_csv(species_cache, sep="\t", low_memory=False)
        missing_now = sorted(set(accs_now) - set(spp["gca_bare"]))
    else:
        spp = pd.DataFrame(columns=["gca_bare", "species"])
        missing_now = accs_now
    if missing_now:
        api_key = os.environ.get("NCBI_API_KEY")
        headers = {"api-key": api_key} if api_key else {}
        # Chunk at 100 to stay under URL/page limits.
        new_rows = []
        for start in range(0, len(missing_now), 100):
            batch = missing_now[start:start + 100]
            r = requests.get(
                f"https://api.ncbi.nlm.nih.gov/datasets/v2/genome/accession/{','.join(batch)}/dataset_report",
                params={"page_size": 200}, headers=headers, timeout=120,
            )
            for rep in r.json().get("reports", []):
                org = rep.get("organism", {}) or {}
                new_rows.append({
                    "gca_bare": str(rep.get("accession", "")).split(".")[0],
                    "species":  org.get("organism_name"),
                })
        spp = pd.concat([spp, pd.DataFrame(new_rows)], ignore_index=True)
        spp.to_csv(species_cache, sep="\t", index=False)
        print(f"Fetched species for {len(new_rows)} accessions; cached at {species_cache.name}")

    out = hicov.merge(spp, on="gca_bare", how="left")
    print(f"\nSpecies breakdown for the {len(out)} qualifying rows:")
    display(out["species"].fillna("(unknown)").value_counts())

    print("\nSample rows:")
    display(out[["gca", "species", "ncbi_n_contigs", "ncbi_contig_n50",
                 "ncbi_genome_coverage", "ncbi_genome_notes", "ncbi_sequencing_tech"]].head(15))

## 5 · Unify the three tiers into one long-format DataFrame

Columns: `tier`, `n_contigs`, `contig_n50`, `completeness`, `contamination`, `lr_coverage_est` (LR tiers only), `level`, `has_gcf` (LR tiers only).

In [ ]:
refseq_df = pd.DataFrame({
    "tier":            "is_refseq",
    "n_contigs":       pd.to_numeric(refseq_q["ncbi_n_contigs"], errors="coerce"),
    "contig_n50":      pd.to_numeric(refseq_q["ncbi_contig_n50"], errors="coerce"),
    "completeness":    pd.to_numeric(refseq_q["ncbi_completeness"], errors="coerce"),
    "contamination":   pd.to_numeric(refseq_q["ncbi_contamination"], errors="coerce"),
    "lr_coverage_est": np.nan,
    "level":           refseq_q["ncbi_assembly_level"].fillna("RefSeq"),
    "has_gcf":         True,
})

lr_df = pd.DataFrame({
    "tier":            np.where(lr_gca_q["is_complete_level"], "LR · Complete Genome", "LR · non-complete"),
    "n_contigs":       pd.to_numeric(lr_gca_q["ncbi_n_contigs"], errors="coerce"),
    "contig_n50":      pd.to_numeric(lr_gca_q["ncbi_contig_n50"], errors="coerce"),
    "completeness":    pd.to_numeric(lr_gca_q["ncbi_completeness"], errors="coerce"),
    "contamination":   pd.to_numeric(lr_gca_q["ncbi_contamination"], errors="coerce"),
    "lr_coverage_est": pd.to_numeric(lr_gca_q["lr_coverage_est"], errors="coerce"),
    "level":           lr_gca_q["level"],
    "has_gcf":         lr_gca_q["has_gcf"],
})

tiers = pd.concat([refseq_df, lr_df], ignore_index=True)
print("Rows per tier (unified, NCBI-sourced QC):")
print(tiers["tier"].value_counts().reindex(TIER_ORDER))
print("\nNon-null counts per metric × tier:")
print(tiers.groupby("tier")[["n_contigs", "contig_n50", "completeness", "contamination", "lr_coverage_est"]].apply(lambda d: d.notna().sum()).reindex(TIER_ORDER))

# ─── Direction-aware cutoff helper (used by §6 histograms, §9b Table A, §9c) ───
# For each refseq_q column, does a LOW value mean BAD quality?
LOW_IS_WORST = {
    "ncbi_completeness":  True,   # low completeness  → bad → cutoff is a FLOOR  (quantile p)
    "ncbi_contig_n50":    True,   # low N50           → bad → cutoff is a FLOOR
    "ncbi_contamination": False,  # high contamination→ bad → cutoff is a CEILING (quantile 1-p)
    "ncbi_n_contigs":     False,  # high contig count → bad → cutoff is a CEILING
}

def refseq_cutoff(col: str, trim_pct: float) -> float:
    """RefSeq cutoff for `col` after trimming the worst `trim_pct`% of RefSeq.

    trim_pct=0 → empirical extreme (most permissive).
    trim_pct=1 → cutoff after dropping the worst 1 % of RefSeq.
    """
    p = trim_pct / 100.0
    q = p if LOW_IS_WORST[col] else (1 - p)
    return float(refseq_q[col].quantile(q))

# Unified-metric name → refseq_q column (for histogram annotation).
METRIC_TO_REFSEQ = {
    "completeness":  "ncbi_completeness",
    "contig_n50":    "ncbi_contig_n50",
    "contamination": "ncbi_contamination",
    "n_contigs":     "ncbi_n_contigs",
}

## 6 · Per-metric histograms (3 tiers overlaid)

In [ ]:
def overlay_hist(metric, *, log_x=False, bins=60, xlim=None, title=None, restrict_tiers=None):
    fig, ax = plt.subplots(figsize=(8, 4))
    tier_subset = restrict_tiers or TIER_ORDER
    for t in tier_subset:
        vals = tiers.loc[tiers["tier"] == t, metric].dropna()
        if vals.empty:
            continue
        colour, alpha = TIER_COLOURS[t]
        if log_x:
            vals = vals.loc[vals > 0]
            if vals.empty:
                continue
            edges = np.logspace(np.log10(vals.min()), np.log10(vals.max()), bins)
        else:
            edges = bins
        ax.hist(vals, bins=edges, alpha=alpha, color=colour, label=f"{t} (n={len(vals)})")

    # Red = worst RefSeq value (cutoff at USE_REFSEQ_PERCENTILE=0).
    # Orange = worst after trimming the worst 1 % of RefSeq.
    if metric in METRIC_TO_REFSEQ:
        rs_col = METRIC_TO_REFSEQ[metric]
        worst    = refseq_cutoff(rs_col, 0)
        worst_1p = refseq_cutoff(rs_col, 1)
        ax.axvline(worst,    color="red",    linestyle="--", linewidth=1.4, alpha=0.9, label=f"worst RefSeq = {worst:.4g}")
        ax.axvline(worst_1p, color="orange", linestyle="--", linewidth=1.4, alpha=0.9, label=f"worst_1pct = {worst_1p:.4g}")

    if log_x:
        ax.set_xscale("log")
    if xlim:
        ax.set_xlim(*xlim)
    ax.set_xlabel(metric)
    ax.set_ylabel("count")
    ax.set_title(title or metric)
    ax.legend(loc="best", fontsize=8)
    fig.tight_layout()
    plt.show()

overlay_hist("contig_n50",    log_x=True,  title="contig_n50 (NCBI Datasets, all tiers)")
overlay_hist("n_contigs",     log_x=True,  title="n_contigs (NCBI Datasets, all tiers)")
overlay_hist("completeness",  log_x=False, bins=80, xlim=(80, 101), title="CheckM completeness (NCBI v1, zoomed 80–100)")
overlay_hist("contamination", log_x=False, bins=80, xlim=(0, 10),   title="CheckM contamination (NCBI v1, zoomed 0–10)")

## 7 · LR run coverage (LR tiers only)

`base_count / 5,500,000` — proxy for how deeply the long reads cover a 5.5 Mb Klebsiella genome.

In [ ]:
overlay_hist("lr_coverage_est", log_x=True, bins=60,
             title="Estimated LR coverage (×, log)",
             restrict_tiers=["LR · Complete Genome", "LR · non-complete"])

## 8 · Bivariate scatter — `n_contigs` vs `contig_n50`

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for t in TIER_ORDER:
    sub = tiers.loc[tiers["tier"] == t].dropna(subset=["n_contigs", "contig_n50"])
    colour, alpha = TIER_COLOURS[t]
    ax.scatter(sub["n_contigs"], sub["contig_n50"], s=8, alpha=alpha, color=colour, label=f"{t} (n={len(sub)})")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("n_contigs (log)"); ax.set_ylabel("contig_n50 (log)")
ax.set_title("n_contigs vs contig_n50, by tier")
ax.legend(loc="best", fontsize=9)
fig.tight_layout()
plt.show()

## 9 · Stacked bar — `assembly_level` × `has_GCF` (LR GCAs only)

Having a paired RefSeq GCF is a strong 'NCBI thought this was reference-tier' signal independent of assembly level.

In [ ]:
lvl_order = ["Complete Genome", "Chromosome", "Scaffold", "Contig"]
ct = (
    lr_gca.assign(level=pd.Categorical(lr_gca["level"], categories=lvl_order, ordered=True))
          .groupby(["level", "has_gcf"], observed=False).size().unstack(fill_value=0)
)
ct = ct.reindex(columns=[True, False], fill_value=0)
fig, ax = plt.subplots(figsize=(7, 4))
ct.plot(kind="bar", stacked=True, ax=ax, color=["#4a7ab4", "#bbbbbb"])
ax.set_xlabel("assembly level"); ax.set_ylabel("count")
ax.set_title("LR-GCAs by assembly level, stacked by paired-GCF (True=RefSeq curated)")
ax.legend(title="has_gcf", labels=["True", "False"])
fig.tight_layout()
plt.show()
display(ct)

## 9b · Table A — RefSeq cutoff candidates

Per-metric, direction-aware cutoffs. The three rows correspond to the three plausible values of `USE_REFSEQ_PERCENTILE`:

- **`worst`** — the empirical extreme. Cutoff if we exclude **nothing** from RefSeq. For `completeness` / `contig_n50` this is the minimum; for `contamination` / `n_contigs` it's the maximum.
- **`worst_1pct`** — cutoff after dropping the **worst 1 %** of RefSeq.
- **`worst_2pct`** — cutoff after dropping the **worst 2 %** of RefSeq.

The red and orange dashed lines on the §6 histograms correspond to `worst` and `worst_1pct` respectively, so you can eyeball where the cutoff sits in each distribution.

In [ ]:
TRIM_PCTS = [0, 1, 2]
ROW_LABELS = ["worst", "worst_1pct", "worst_2pct"]

table_a = pd.DataFrame(
    {col: [refseq_cutoff(col, p) for p in TRIM_PCTS] for col in LOW_IS_WORST},
    index=ROW_LABELS,
).round(2)

# Annotate each column with its direction (floor or ceiling).
table_a.columns = [
    f"{c}\n({'≥ floor' if LOW_IS_WORST[c] else '≤ ceiling'})"
    for c in table_a.columns
]

print("=== Table A — RefSeq cutoff candidates (direction-aware) ===")
print(f"(based on {refseq_q['ncbi_completeness'].notna().sum()} is_refseq rows with NCBI CheckM)\n")
display(table_a)

## 9c · Resolve thresholds from the RefSeq envelope

Take the user's `USE_REFSEQ_PERCENTILE` (defined in the parameters cell at the top) and pull the matching floor/ceiling out of Table A. By construction, this rule excludes zero is_refseq genomes at percentile 0; bumping the percentile to 1 or 2 trims the most extreme RefSeq outliers (and excludes them too — quantified in §10).

In [ ]:
assert 0 <= USE_REFSEQ_PERCENTILE < 50, "USE_REFSEQ_PERCENTILE should be in [0, 50)"

MIN_COMPLETENESS  = refseq_cutoff("ncbi_completeness",  USE_REFSEQ_PERCENTILE)
MAX_CONTAMINATION = refseq_cutoff("ncbi_contamination", USE_REFSEQ_PERCENTILE)
MAX_N_CONTIGS     = refseq_cutoff("ncbi_n_contigs",     USE_REFSEQ_PERCENTILE)
MIN_CONTIG_N50    = refseq_cutoff("ncbi_contig_n50",    USE_REFSEQ_PERCENTILE)

print(f"USE_REFSEQ_PERCENTILE = {USE_REFSEQ_PERCENTILE}  (trim worst {USE_REFSEQ_PERCENTILE}% of RefSeq)")
print(f"  MIN_COMPLETENESS  ≥ {MIN_COMPLETENESS:.2f}")
print(f"  MAX_CONTAMINATION ≤ {MAX_CONTAMINATION:.2f}")
print(f"  MAX_N_CONTIGS     ≤ {MAX_N_CONTIGS:.0f}")
print(f"  MIN_CONTIG_N50    ≥ {MIN_CONTIG_N50:.0f}")
print(f"  MIN_LR_COVERAGE   ≥ {MIN_LR_COVERAGE:.0f}×   (LR-only; no RefSeq comparator)")

## 10 · Apply the data-driven LRA rule — Table B (per-tier exclusions)

Rule, in order:

```
if CheckM completeness is NaN   → reject (NCBI flagged anomalous)
elif completeness  < MIN_COMPLETENESS    → reject
elif contamination > MAX_CONTAMINATION   → reject
elif n_contigs     > MAX_N_CONTIGS       → reject
elif contig_n50    < MIN_CONTIG_N50      → reject
elif lr_coverage_est present AND  < MIN_LR_COVERAGE → reject
else → accept
```

Reported below:
- **Table B-1** — per-criterion exclusion counts, split by tier (is_refseq / LR Complete / LR non-complete).
- **Table B-2** — accept vs reject totals per tier and accept rate.
- **Sample** — 15 randomly rejected `LR · Complete Genome` rows so you can eyeball what's being discarded among the assemblies that *should* be safe.

In [ ]:
def classify(row):
    """Apply the data-driven LRA rule; return (accept_bool, reason_str)."""
    if pd.isna(row["completeness"]):
        return False, "checkm_missing"
    if row["completeness"] < MIN_COMPLETENESS:
        return False, "completeness"
    if pd.notna(row["contamination"]) and row["contamination"] > MAX_CONTAMINATION:
        return False, "contamination"
    if pd.notna(row["n_contigs"]) and row["n_contigs"] > MAX_N_CONTIGS:
        return False, "n_contigs"
    if pd.notna(row["contig_n50"]) and row["contig_n50"] < MIN_CONTIG_N50:
        return False, "contig_n50"
    if pd.notna(row["lr_coverage_est"]) and row["lr_coverage_est"] < MIN_LR_COVERAGE:
        return False, "lr_coverage"
    return True, "accept"

verdicts = tiers.apply(classify, axis=1, result_type="expand")
verdicts.columns = ["accept", "reason"]
tiers_v = pd.concat([tiers, verdicts], axis=1)

# Table B-1 — per-criterion exclusion counts × tier.
print("=== Table B-1 — exclusions by criterion × tier ===")
b1 = (
    tiers_v.loc[~tiers_v["accept"]]
           .groupby(["tier", "reason"]).size().unstack(fill_value=0)
           .reindex(index=TIER_ORDER)
           .reindex(columns=["checkm_missing", "completeness", "contamination", "n_contigs", "contig_n50", "lr_coverage"], fill_value=0)
)
b1["TOTAL_REJECTED"] = b1.sum(axis=1)
b1["tier_n"] = tiers_v.groupby("tier").size().reindex(TIER_ORDER).values
display(b1)

# Table B-2 — accept / reject totals + accept rate.
print("\n=== Table B-2 — accept vs reject per tier ===")
b2 = (
    tiers_v.groupby(["tier", "accept"]).size().unstack(fill_value=0)
           .reindex(index=TIER_ORDER)
           .rename(columns={True: "accept", False: "reject"})
)
b2["total"]       = b2["accept"] + b2["reject"]
b2["accept_rate"] = (b2["accept"] / b2["total"]).round(3)
display(b2)

# Sample of rejected Complete-Genome LR rows (what are we throwing away among the safe-by-construction set?).
print("\n=== Sample (n=15) of REJECTED LR · Complete Genome rows ===")
rej_complete = tiers_v[(tiers_v["tier"] == "LR · Complete Genome") & (~tiers_v["accept"])]
print(f"Total rejected Complete-Genome LR-GCAs: {len(rej_complete)} / 1202")
display(rej_complete[["reason", "completeness", "contamination", "n_contigs", "contig_n50", "lr_coverage_est"]].sample(min(15, len(rej_complete)), random_state=0))

## 11 · Locked decision — paste into `build_lra_set.py`

Once happy with the thresholds above, fill the cell below in plain text (a markdown record, not code) for future readers, then copy the constants into the module.

**LRA acceptance rule (LOCKED):** _fill in after iterating in §0 / §9c / §10_

- `USE_REFSEQ_PERCENTILE = …`
- `MIN_LR_COVERAGE = …`

Resolved thresholds (printed by §9c):
- `MIN_COMPLETENESS  ≥ …`
- `MAX_CONTAMINATION ≤ …`
- `MAX_N_CONTIGS     ≤ …`
- `MIN_CONTIG_N50    ≥ …`

**Accepted LRA set size:** _LR Complete + LR non-complete that pass = N_ (from Table B-2)

**Rationale:** _one paragraph — why this percentile, why this coverage floor, what the rejected Complete-Genome sample looked like_